In [1]:
from gc import callbacks

from chats import llm_streaming

/Users/ikartavost-pc/pet_projects/ai/langchain-course/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [2]:
tokens = []

async for token in llm_streaming.astream("What is NLP?"):
    tokens.append(token)
    print(token.content, end="|", flush=True)

||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||**|Natural| Language| Processing| (|N|LP|)**| is| a| branch| of| **|art|ificial| intelligence| (|AI|)**| that| focuses| on| enabling| computers| to| **|under|stand|,| interpret|,| and| generate| human| language|**| in| a| way| that| is| both| meaningful| and| useful|.| It| bridges| the| gap| between| human| communication| and| machine| understanding|,| allowing| technology| to| process| and| respond| to| natural| language| input| (|like| text| or| speech|)| in| a| human|-like| manner|.

|---

|###| **|Key| Components| of| N|LP|**
|1|.| **|Text| Analysis|**:|  
|  | -| Ident|ifying| patterns|,| sentime

In [3]:
tokens[0]

AIMessageChunk(content='', additional_kwargs={}, response_metadata={}, id='lc_run--019cc850-5ef7-7a73-be64-48212d16d5c9', tool_calls=[], invalid_tool_calls=[], tool_call_chunks=[])

In [11]:
tokens[956]

AIMessageChunk(content=' contract', additional_kwargs={}, response_metadata={}, id='lc_run--019cc850-5ef7-7a73-be64-48212d16d5c9', tool_calls=[], invalid_tool_calls=[], tool_call_chunks=[])

In [12]:
from langchain_core.tools import tool

@tool
def add(x: float, y: float) -> float:
    """Add 'x' and 'y'."""
    return x + y

@tool
def multiply(x: float, y: float) -> float:
    """Multiply 'x' and 'y'."""
    return x * y

@tool
def exponentiate(x: float, y: float) -> float:
    """Raise 'x' to the power of 'y'."""
    return x ** y

@tool
def subtract(x: float, y: float) -> float:
    """Subtract 'x' from 'y'."""
    return y - x

@tool
def final_answer(answer: str, tools_used: list[str]) -> str:
    """Use this tool to provide a final answer to the user.
    The answer should be in natural language as this will be provided
    to the user directly. The tools_used must include a list of tool
    names that were used within the `scratchpad`. You MUST use this tool
    to conclude the interaction.
    """
    return {"answer": answer, "tools_used": tools_used}

In [13]:
tools = [add, multiply, exponentiate, subtract, final_answer]

In [15]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages([
    ("system", (
        "You're a helpful assistant. When answering a user's question "
        "you should first use one of the tools provided. After using a "
        "tool the tool output will be provided back to you. You MUST "
        "then use the final_answer tool to provide a final answer to the user. "
        "DO NOT use the same tool more than once."
    )),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

In [16]:
from langchain_core.runnables.base import RunnableSerializable

tools = [add, subtract, multiply, exponentiate, final_answer]

# define the agent runnable
agent: RunnableSerializable = (
    {
        "input": lambda x: x["input"],
        "chat_history": lambda x: x["chat_history"],
        "agent_scratchpad": lambda x: x.get("agent_scratchpad", [])
    }
    | prompt
    | llm_streaming.bind_tools(tools, tool_choice="any")
)

In [17]:
import json
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage


# create tool name to function mapping
name2tool = {tool.name: tool.func for tool in tools}

class CustomAgentExecutor:
    chat_history: list[BaseMessage]

    def __init__(self, max_iterations: int = 3):
        self.chat_history = []
        self.max_iterations = max_iterations
        self.agent: RunnableSerializable = (
            {
                "input": lambda x: x["input"],
                "chat_history": lambda x: x["chat_history"],
                "agent_scratchpad": lambda x: x.get("agent_scratchpad", [])
            }
            | prompt
            | llm_streaming.bind_tools(tools, tool_choice="any")  # we're forcing tool use again
        )

    def invoke(self, input: str) -> dict:
        # invoke the agent but we do this iteratively in a loop until
        # reaching a final answer
        count = 0
        agent_scratchpad = []
        while count < self.max_iterations:
            # invoke a step for the agent to generate a tool call
            out = self.agent.invoke({
                "input": input,
                "chat_history": self.chat_history,
                "agent_scratchpad": agent_scratchpad
            })
            # if the tool call is the final answer tool, we stop
            if out.tool_calls[0]["name"] == "final_answer":
                break
            agent_scratchpad.append(out)  # add tool call to scratchpad
            # otherwise we execute the tool and add it's output to the agent scratchpad
            tool_out = name2tool[out.tool_calls[0]["name"]](**out.tool_calls[0]["args"])
            # add the tool output to the agent scratchpad
            action_str = f"The {out.tool_calls[0]['name']} tool returned {tool_out}"
            agent_scratchpad.append({
                "role": "tool",
                "content": action_str,
                "tool_call_id": out.tool_calls[0]["id"]
            })
            # add a print so we can see intermediate steps
            print(f"{count}: {action_str}")
            count += 1
        # add the final output to the chat history
        final_answer = out.tool_calls[0]["args"]
        # this is a dictionary, so we convert it to a string for compatibility with
        # the chat history
        final_answer_str = json.dumps(final_answer)
        self.chat_history.append({"input": input, "output": final_answer_str})
        self.chat_history.extend([
            HumanMessage(content=input),
            AIMessage(content=final_answer_str)
        ])
        # return the final answer in dict form
        return final_answer

agent_executor = CustomAgentExecutor()

In [18]:
agent_executor.invoke(input="What is 10 + 10")

0: The add tool returned 20


{'answer': 'The result of 10 + 10 is 20.', 'tools_used': ['add']}

In [19]:
from langchain_core.runnables import  ConfigurableField

llm_conf = llm_streaming.configurable_fields(
    callbacks=ConfigurableField(
        id="callbacks",
        name="callbacks",
        description="A list of callbacks to use for streaming"
    )
)


In [20]:
# define the agent runnable
agent: RunnableSerializable = (
    {
        "input": lambda x: x["input"],
        "chat_history": lambda x: x["chat_history"],
        "agent_scratchpad": lambda x: x.get("agent_scratchpad", [])
    }
    | prompt
    | llm_conf.bind_tools(tools, tool_choice="any")
)

In [21]:
import asyncio
from langchain_core.callbacks.base import AsyncCallbackHandler


class QueueCallbackHandler(AsyncCallbackHandler):
    """Callback handler that puts tokens into a queue."""

    def __init__(self, queue: asyncio.Queue):
        self.queue = queue
        self.final_answer_seen = False

    async def __aiter__(self):
        while True:
            if self.queue.empty():
                await asyncio.sleep(0.1)
                continue
            token_or_done = await self.queue.get()

            if token_or_done == "<<DONE>>":
                # this means we're done
                return
            if token_or_done:
                yield token_or_done

    async def on_llm_new_token(self, *args, **kwargs) -> None:
        """Put new token in the queue."""
        #print(f"on_llm_new_token: {args}, {kwargs}")
        chunk = kwargs.get("chunk")
        if chunk:
            # check for final_answer tool call
            if tool_calls := chunk.message.additional_kwargs.get("tool_calls"):
                if tool_calls[0]["function"]["name"] == "final_answer":
                    # this will allow the stream to end on the next `on_llm_end` call
                    self.final_answer_seen = True
        await self.queue.put(chunk)
        return

    async def on_llm_end(self, *args, **kwargs) -> None:
        """Put None in the queue to signal completion."""
        #print(f"on_llm_end: {args}, {kwargs}")
        # this should only be used at the end of our agent execution, however LangChain
        # will call this at the end of every tool call, not just the final tool call
        # so we must only send the "done" signal if we have already seen the final_answer
        # tool call
        if self.final_answer_seen:
            await self.queue.put("<<DONE>>")
        else:
            await self.queue.put("<<STEP_END>>")
        return

In [22]:
queue = asyncio.Queue()
streamer = QueueCallbackHandler(queue)

In [24]:
tokens = []

async def stream(query: str):
    response = agent.with_config(
        callbacks=[streamer]
    )
    async for token in response.astream({
        "input": query,
        "chat_history": [],
        "agent_scratchpad": []
    }):
        tokens.append(token)
        print(token, flush=True)

await stream("What is 10 + 10")

content='' additional_kwargs={} response_metadata={} id='lc_run--019cc868-487a-7261-a4f9-745f9a6de1ba' tool_calls=[] invalid_tool_calls=[] tool_call_chunks=[]
content='' additional_kwargs={} response_metadata={} id='lc_run--019cc868-487a-7261-a4f9-745f9a6de1ba' tool_calls=[] invalid_tool_calls=[] tool_call_chunks=[]
content='' additional_kwargs={} response_metadata={} id='lc_run--019cc868-487a-7261-a4f9-745f9a6de1ba' tool_calls=[] invalid_tool_calls=[] tool_call_chunks=[]
content='' additional_kwargs={} response_metadata={} id='lc_run--019cc868-487a-7261-a4f9-745f9a6de1ba' tool_calls=[] invalid_tool_calls=[] tool_call_chunks=[]
content='' additional_kwargs={} response_metadata={} id='lc_run--019cc868-487a-7261-a4f9-745f9a6de1ba' tool_calls=[] invalid_tool_calls=[] tool_call_chunks=[]
content='' additional_kwargs={} response_metadata={} id='lc_run--019cc868-487a-7261-a4f9-745f9a6de1ba' tool_calls=[] invalid_tool_calls=[] tool_call_chunks=[]
content='' additional_kwargs={} response_metad

In [25]:
tokens

[AIMessageChunk(content='', additional_kwargs={}, response_metadata={}, id='lc_run--019cc868-487a-7261-a4f9-745f9a6de1ba', tool_calls=[], invalid_tool_calls=[], tool_call_chunks=[]),
 AIMessageChunk(content='', additional_kwargs={}, response_metadata={}, id='lc_run--019cc868-487a-7261-a4f9-745f9a6de1ba', tool_calls=[], invalid_tool_calls=[], tool_call_chunks=[]),
 AIMessageChunk(content='', additional_kwargs={}, response_metadata={}, id='lc_run--019cc868-487a-7261-a4f9-745f9a6de1ba', tool_calls=[], invalid_tool_calls=[], tool_call_chunks=[]),
 AIMessageChunk(content='', additional_kwargs={}, response_metadata={}, id='lc_run--019cc868-487a-7261-a4f9-745f9a6de1ba', tool_calls=[], invalid_tool_calls=[], tool_call_chunks=[]),
 AIMessageChunk(content='', additional_kwargs={}, response_metadata={}, id='lc_run--019cc868-487a-7261-a4f9-745f9a6de1ba', tool_calls=[], invalid_tool_calls=[], tool_call_chunks=[]),
 AIMessageChunk(content='', additional_kwargs={}, response_metadata={}, id='lc_run--0

In [27]:
from langchain_core.messages import ToolMessage

class CustomAgentExecutor:
    chat_history: list[BaseMessage]

    def __init__(self, max_iterations: int = 3):
        self.chat_history = []
        self.max_iterations = max_iterations
        self.agent: RunnableSerializable = (
            {
                "input": lambda x: x["input"],
                "chat_history": lambda x: x["chat_history"],
                "agent_scratchpad": lambda x: x.get("agent_scratchpad", [])
            }
            | prompt
            | llm_conf.bind_tools(tools, tool_choice="any")  # we're forcing tool use again
        )

    async def invoke(self, input: str, streamer: QueueCallbackHandler, verbose: bool = False) -> dict:
        # invoke the agent but we do this iteratively in a loop until
        # reaching a final answer
        count = 0
        agent_scratchpad = []
        while count < self.max_iterations:
            # invoke a step for the agent to generate a tool call
            async def stream(query: str):
                response = self.agent.with_config(
                    callbacks=[streamer]
                )
                # we initialize the output dictionary that we will be populating with
                # our streamed output
                output = None
                # now we begin streaming
                async for token in response.astream({
                    "input": query,
                    "chat_history": self.chat_history,
                    "agent_scratchpad": agent_scratchpad
                }):
                    if output is None:
                        output = token
                    else:
                        # we can just add the tokens together as they are streamed and
                        # we'll have the full response object at the end
                        output += token
                    if token.content != "":
                        # we can capture various parts of the response object
                        if verbose: print(f"content: {token.content}", flush=True)
                    tool_calls = token.additional_kwargs.get("tool_calls")
                    if tool_calls:
                        if verbose: print(f"tool_calls: {tool_calls}", flush=True)
                        tool_name = tool_calls[0]["function"]["name"]
                        if tool_name:
                            if verbose: print(f"tool_name: {tool_name}", flush=True)
                        arg = tool_calls[0]["function"]["arguments"]
                        if arg != "":
                            if verbose: print(f"arg: {arg}", flush=True)
                return AIMessage(
                    content=output.content,
                    tool_calls=output.tool_calls,
                    tool_call_id=output.tool_calls[0]["id"]
                )

            tool_call = await stream(query=input)
            # add initial tool call to scratchpad
            agent_scratchpad.append(tool_call)
            # otherwise we execute the tool and add it's output to the agent scratchpad
            tool_name = tool_call.tool_calls[0]["name"]
            tool_args = tool_call.tool_calls[0]["args"]
            tool_call_id = tool_call.tool_call_id
            tool_out = name2tool[tool_name](**tool_args)
            # add the tool output to the agent scratchpad
            tool_exec = ToolMessage(
                content=f"{tool_out}",
                tool_call_id=tool_call_id
            )
            agent_scratchpad.append(tool_exec)
            count += 1
            # if the tool call is the final answer tool, we stop
            if tool_name == "final_answer":
                break
        # add the final output to the chat history, we only add the "answer" field
        final_answer = tool_out["answer"]
        self.chat_history.extend([
            HumanMessage(content=input),
            AIMessage(content=final_answer)
        ])
        # return the final answer in dict form
        return tool_args

agent_executor = CustomAgentExecutor()

In [31]:
queue = asyncio.Queue()
streamer = QueueCallbackHandler(queue)

out = await agent_executor.invoke("What is 10 + 10", streamer, verbose=True)

In [32]:
out

{'answer': '20', 'tools_used': ['add']}

In [33]:
queue = asyncio.Queue()
streamer = QueueCallbackHandler(queue)

task = asyncio.create_task(agent_executor.invoke("What is 10 + 10", streamer))

async for token in streamer:
    print(token, flush=True)

await task

message=AIMessageChunk(content='', additional_kwargs={}, response_metadata={}, id='lc_run--019cc872-bc48-7180-9a54-ae62f18ad247', tool_calls=[], invalid_tool_calls=[], tool_call_chunks=[])
message=AIMessageChunk(content='', additional_kwargs={}, response_metadata={}, id='lc_run--019cc872-bc48-7180-9a54-ae62f18ad247', tool_calls=[], invalid_tool_calls=[], tool_call_chunks=[])
message=AIMessageChunk(content='', additional_kwargs={}, response_metadata={}, id='lc_run--019cc872-bc48-7180-9a54-ae62f18ad247', tool_calls=[], invalid_tool_calls=[], tool_call_chunks=[])
message=AIMessageChunk(content='', additional_kwargs={}, response_metadata={}, id='lc_run--019cc872-bc48-7180-9a54-ae62f18ad247', tool_calls=[], invalid_tool_calls=[], tool_call_chunks=[])
message=AIMessageChunk(content='', additional_kwargs={}, response_metadata={}, id='lc_run--019cc872-bc48-7180-9a54-ae62f18ad247', tool_calls=[], invalid_tool_calls=[], tool_call_chunks=[])
message=AIMessageChunk(content='', additional_kwargs={}

CancelledError: 

In [ ]:
queue = asyncio.Queue()
streamer = QueueCallbackHandler(queue)

task = asyncio.create_task(agent_executor.invoke("What is 10 + 10", streamer))

async for token in streamer:
    # first identify if we have a <<STEP_END>> token
    if token == "<<STEP_END>>":
        print("\n", flush=True)
    # we'll first identify if the token is a tool call
    elif tool_calls := token.message.additional_kwargs.get("tool_calls"):
        # if we have a tool call with a tool name, we'll print it
        if tool_name := tool_calls[0]["function"]["name"]:
            print(f"Calling {tool_name}...", flush=True)
        # if we have a tool call with arguments, we ad them to our args string
        if tool_args := tool_calls[0]["function"]["arguments"]:
            print(f"{tool_args}", end="", flush=True)

_ = await task

Task exception was never retrieved
future: <Task finished name='Task-4122' coro=<CustomAgentExecutor.invoke() done, defined at /var/folders/dc/v08dhs892w5ffjzk9_wrf6800000gn/T/ipykernel_64991/1598195974.py:19> exception=IndexError('list index out of range')>
Traceback (most recent call last):
  File "/var/folders/dc/v08dhs892w5ffjzk9_wrf6800000gn/T/ipykernel_64991/1598195974.py", line 63, in invoke
    tool_call = await stream(query=input)
                ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/var/folders/dc/v08dhs892w5ffjzk9_wrf6800000gn/T/ipykernel_64991/1598195974.py", line 60, in stream
    tool_call_id=output.tool_calls[0]["id"]
                 ~~~~~~~~~~~~~~~~~^^^
IndexError: list index out of range
